In [ ]:
# ==============================================================================
# # Install Libraries
# ==============================================================================
!pip install -q torch torchvision torchaudio numpy matplotlib pandas

# ==============================================================================
# # Imports
# ==============================================================================
import os
import time
import random
import collections
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ==============================================================================
# # Configuration
# ==============================================================================
class Config:
    # Environment Configurations
    MAZE_SIZE = 10          # Options: 10, 15, 20, 30
    WALL_PROBABILITY = 0.2  # Probability of a cell being a wall

    # MODIFIED: Ab steps ki koi limit nahi hai, jab tak solve nahi hoga chalta rahega!
    MAX_STEPS = float('inf')

    # RL Hyperparameters
    GAMMA = 0.99            # Discount factor
    LEARNING_RATE = 0.0005  # Learning rate for Adam optimizer
    MEMORY_SIZE = 20000     # Replay buffer capacity
    BATCH_SIZE = 64         # Mini-batch size for training
    TARGET_UPDATE = 5       # Target network update frequency (in episodes)

    # Exploration Settings
    EPSILON_START = 1.0     # Initial exploration rate
    EPSILON_END = 0.05      # Minimum exploration rate
    EPSILON_DECAY = 0.99    # Multiplicative decay factor per episode

    # Training Control
    NUM_EPISODES = 250      # Total training episodes

    # Device Configuration
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {Config.DEVICE}")

# ==============================================================================
# # Maze Generator
# ==============================================================================
class MazeGenerator:
    """Generates random valid mazes where Start and Goal are guaranteed to be reachable."""

    @staticmethod
    def is_solvable(maze, start, goal):
        size = maze.shape[0]
        queue = collections.deque([start])
        visited = {start}

        while queue:
            curr = queue.popleft()
            if curr == goal:
                return True
            for r_off, c_off in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = curr[0] + r_off, curr[1] + c_off
                if 0 <= nr < size and 0 <= nc < size:
                    if maze[nr, nc] != 1 and (nr, nc) not in visited:
                        visited.add((nr, nc))
                        queue.append((nr, nc))
        return False

    @classmethod
    def generate(cls, size, wall_prob):
        while True:
            maze = np.random.choice([0, 1], size=(size, size), p=[1 - wall_prob, wall_prob])
            all_coords = [(r, c) for r in range(size) for c in range(size)]
            start, goal = random.sample(all_coords, 2)

            maze[start[0], start[1]] = 0
            maze[goal[0], goal[1]] = 0

            if cls.is_solvable(maze, start, goal):
                return maze, start, goal

# ==============================================================================
# # Environment
# ==============================================================================
class MazeEnvironment:
    """Custom Reinforcement Learning Grid Environment for Maze Solving."""
    def __init__(self, size=Config.MAZE_SIZE, wall_prob=Config.WALL_PROBABILITY):
        self.size = size
        self.wall_prob = wall_prob
        self.maze = None
        self.start_pos = None
        self.goal_pos = None
        self.agent_pos = None
        self.steps = 0
        self.visited_cells = set()

        self.wall_collisions = 0
        self.wrong_moves = 0
        self.revisited_count = 0
        self.action_space_size = 4

    def reset(self):
        self.maze, self.start_pos, self.goal_pos = MazeGenerator.generate(self.size, self.wall_prob)
        self.agent_pos = self.start_pos
        self.steps = 0
        self.visited_cells = {self.start_pos}
        self.wall_collisions = 0
        self.wrong_moves = 0
        self.revisited_count = 0
        return self.get_state()

    def get_state(self):
        state_tensor = np.zeros((3, self.size, self.size), dtype=np.float32)
        state_tensor[0] = self.maze
        state_tensor[1, self.agent_pos[0], self.agent_pos[1]] = 1.0
        state_tensor[2, self.goal_pos[0], self.goal_pos[1]] = 1.0
        return state_tensor

    def step(self, action):
        self.steps += 1
        r, c = self.agent_pos

        if action == 0:    # Up
            nr, nc = r - 1, c
        elif action == 1:  # Down
            nr, nc = r + 1, c
        elif action == 2:  # Left
            nr, nc = r, c - 1
        elif action == 3:  # Right
            nr, nc = r, c + 1
        else:
            nr, nc = r, c

        prev_dist = abs(r - self.goal_pos[0]) + abs(c - self.goal_pos[1])

        # Check boundaries
        if nr < 0 or nr >= self.size or nc < 0 or nc >= self.size:
            reward = -10
            self.wall_collisions += 1
            done = False
        # Check walls
        elif self.maze[nr, nc] == 1:
            reward = -10
            self.wall_collisions += 1
            done = False
        else:
            self.agent_pos = (nr, nc)
            new_dist = abs(nr - self.goal_pos[0]) + abs(nc - self.goal_pos[1])

            if self.agent_pos == self.goal_pos:
                reward = 100
                done = True
            elif self.agent_pos in self.visited_cells:
                reward = -3
                self.revisited_count += 1
                done = False
            else:
                reward = -1
                self.visited_cells.add(self.agent_pos)
                done = False
                if new_dist >= prev_dist:
                    self.wrong_moves += 1

        # MODIFIED: Max steps termination framework removed entirely.
        # The episode will only end when done == True (Goal reached).
        return self.get_state(), reward, done

# ==============================================================================
# # Replay Buffer
# ==============================================================================
class ReplayBuffer:
    def __init__(self, capacity=Config.MEMORY_SIZE):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

    def __len__(self):
        return len(self.buffer)

# ==============================================================================
# # DQN Model
# ==============================================================================
class DQN(nn.Module):
    def __init__(self, input_shape):
        super(DQN, self).__init__()
        channels, height, width = input_shape

        self.conv = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

        conv_out_size = 64 * height * width

        self.fc = nn.Sequential(
            nn.Linear(conv_out_size, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        )

    def forward(self, x):
        conv_out = self.conv(x)
        return self.fc(conv_out.view(conv_out.size(0), -1))

# ==============================================================================
# # Agent
# ==============================================================================
class DQNAgent:
    def __init__(self, state_shape):
        self.state_shape = state_shape
        self.policy_net = DQN(state_shape).to(Config.DEVICE)
        self.target_net = DQN(state_shape).to(Config.DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=Config.LEARNING_RATE)
        self.memory = ReplayBuffer()
        self.epsilon = Config.EPSILON_START

    def select_action(self, state, eval_mode=False):
        if not eval_mode and random.random() < self.epsilon:
            return random.randint(0, 3)

        with torch.no_grad():
            state_t = torch.tensor(np.expand_dims(state, axis=0), dtype=torch.float32, device=Config.DEVICE)
            q_values = self.policy_net(state_t)
            return q_values.argmax().item()

    def update_policy(self):
        if len(self.memory) < Config.BATCH_SIZE:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(Config.BATCH_SIZE)

        states_t = torch.tensor(states, dtype=torch.float32, device=Config.DEVICE)
        actions_t = torch.tensor(actions, dtype=torch.long, device=Config.DEVICE).unsqueeze(1)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=Config.DEVICE)
        next_states_t = torch.tensor(next_states, dtype=torch.float32, device=Config.DEVICE)
        dones_t = torch.tensor(dones, dtype=torch.float32, device=Config.DEVICE)

        current_q = self.policy_net(states_t).gather(1, actions_t).squeeze(1)

        with torch.no_grad():
            next_q = self.target_net(next_states_t).max(1)[0]
            expected_q = rewards_t + (Config.GAMMA * next_q * (1 - dones_t))

        loss = nn.MSELoss()(current_q, expected_q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()

    def update_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(Config.EPSILON_END, self.epsilon * Config.EPSILON_DECAY)

# ==============================================================================
# # Training Function
# ==============================================================================
def train_agent():
    env = MazeEnvironment()
    agent = DQNAgent((3, Config.MAZE_SIZE, Config.MAZE_SIZE))

    history = {
        'episode': [], 'reward': [], 'loss': [], 'epsilon': [],
        'success': [], 'steps': [], 'avg_reward': []
    }

    reward_window = collections.deque(maxlen=30)
    print("--- Starting Agent Deep Q-Network Training Pipeline ---")

    for ep in range(1, Config.NUM_EPISODES + 1):
        state = env.reset()
        total_reward = 0
        ep_losses = []
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.memory.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward

            loss = agent.update_policy()
            if loss is not None:
                ep_losses.append(loss)

        agent.decay_epsilon()
        if ep % Config.TARGET_UPDATE == 0:
            agent.update_target_network()

        avg_loss = np.mean(ep_losses) if ep_losses else 0.0
        success = 1 if env.agent_pos == env.goal_pos else 0
        reward_window.append(total_reward)
        running_avg_reward = np.mean(reward_window)

        history['episode'].append(ep)
        history['reward'].append(total_reward)
        history['loss'].append(avg_loss)
        history['epsilon'].append(agent.epsilon)
        history['success'].append(success)
        history['steps'].append(env.steps)
        history['avg_reward'].append(running_avg_reward)

        if ep == 1 or ep % 10 == 0:
            print(f"Episode: {ep:3d} | Steps: {env.steps:3d} | Total Reward: {total_reward:6.1f} | "
                  f"Avg Reward: {running_avg_reward:6.1f} | Loss: {avg_loss:6.4f} | "
                  f"Epsilon: {agent.epsilon:.3f} | Goal Reached: {'YES' if success else 'NO'}")

    df_logs = pd.DataFrame(history)
    df_logs.to_csv("training_log.csv", index=False)
    print("-> Training logs saved directly to 'training_log.csv'")

    return agent, history

# ==============================================================================
# # Testing Function
# ==============================================================================
def test_agent(agent, num_tests=3):
    print("\n--- Initializing Inference Testing Mode on Unseen Environments ---")
    env = MazeEnvironment()

    for idx in range(1, num_tests + 1):
        start_time = time.time()
        state = env.reset()
        done = False

        shortest_path_taken = [env.start_pos]

        while not done:
            action = agent.select_action(state, eval_mode=True)
            next_state, reward, done = env.step(action)
            state = next_state
            if env.agent_pos not in shortest_path_taken and reward != -10:
                shortest_path_taken.append(env.agent_pos)

        elapsed_time = time.time() - start_time
        success = (env.agent_pos == env.goal_pos)

        print("\n" + "="*40)
        print(f" TEST RUN #{idx} RESULTS ")
        print("="*40)
        print(f"Number of Steps:        {env.steps}")
        print(f"Total Reward:            {100 - env.steps if success else -env.steps}")
        print(f"Path Length:             {len(shortest_path_taken)}")
        print(f"Number of Wrong Moves:   {env.wrong_moves}")
        print(f"Wall Collisions Count:   {env.wall_collisions}")
        print(f"Revisited Cells Count:   {env.revisited_count}")
        print(f"Time Taken:              {elapsed_time:.4f} seconds")
        print(f"Whether Goal was reached:{' SUCCESS' if success else ' FAILED'}")

        render_text_maze(env.maze, env.start_pos, env.goal_pos, shortest_path_taken)

# ==============================================================================
# # Visualization
# ==============================================================================
def render_text_maze(maze, start, goal, path):
    size = maze.shape[0]
    path_set = set(path)

    print("\n[Solved Maze Visualization Blueprint]")
    for r in range(size):
        row_str = ""
        for c in range(size):
            if (r, c) == start:
                row_str += "S"
            elif (r, c) == goal:
                row_str += "G"
            elif (r, c) in path_set:
                row_str += "*"
            elif maze[r, c] == 1:
                row_str += "#"
            else:
                row_str += "."
        print(row_str)

# ==============================================================================
# # Graphs
# ==============================================================================
def plot_performance_graphs(history):
    episodes = history['episode']

    # 1. Episode vs Reward
    plt.figure(figsize=(10, 5))
    plt.plot(episodes, history['reward'], label='Total Reward', color='blue', alpha=0.4)
    plt.plot(episodes, history['avg_reward'], label='Moving Average (30 Ep)', color='red', linewidth=2)
    plt.title('Training Convergence: Episode vs Reward')
    plt.xlabel('Episodes')
    plt.ylabel('Rewards')
    plt.grid(True)
    plt.legend()
    plt.savefig('reward_graph.png')
    plt.show()

    # 2. Episode vs Loss
    plt.figure(figsize=(10, 5))
    plt.plot(episodes, history['loss'], color='purple')
    plt.title('Training Evaluation: Episode vs Loss Optimization')
    plt.xlabel('Episodes')
    plt.ylabel('Loss Metrics')
    plt.grid(True)
    plt.savefig('loss_graph.png')
    plt.show()

    # 3. Success Rate Graph
    plt.figure(figsize=(10, 5))
    success_series = pd.Series(history['success'])
    rolling_success = success_series.rolling(window=20, min_periods=1).mean() * 100
    plt.plot(episodes, rolling_success, color='green', linewidth=2)
    plt.title('Agent Learning Curve: Episode vs Success Rate % (Window=20)')
    plt.xlabel('Episodes')
    plt.ylabel('Success Percentage (%)')
    plt.grid(True)
    plt.savefig('success_graph.png')
    plt.show()

    print("-> Statistical analysis visual plots exported as PNG files safely.")

# ==============================================================================
# # Save Model
# ==============================================================================
def save_trained_model(agent, path="trained_model.pth"):
    torch.save(agent.policy_net.state_dict(), path)
    print(f"-> Complete Neural model state weight parameters exported to '{path}'")

# ==============================================================================
# # Main Function
# ==============================================================================
def main():
    trained_agent, training_history = train_agent()
    plot_performance_graphs(training_history)
    save_trained_model(trained_agent)
    test_agent(trained_agent, num_tests=3)

if __name__ == "__main__":
    main()

Using device: cpu
--- Starting Agent Deep Q-Network Training Pipeline ---
Episode:   1 | Steps: 351 | Total Reward: -1513.0 | Avg Reward: -1513.0 | Loss: 7.0061 | Epsilon: 0.990 | Goal Reached: YES
Episode:  10 | Steps: 122 | Total Reward: -390.0 | Avg Reward: -653.8 | Loss: 21.7637 | Epsilon: 0.904 | Goal Reached: YES
Episode:  20 | Steps: 884 | Total Reward: -3948.0 | Avg Reward: -1357.3 | Loss: 7.3396 | Epsilon: 0.818 | Goal Reached: YES
Episode:  30 | Steps: 4311 | Total Reward: -22267.0 | Avg Reward: -2576.8 | Loss: 5.7313 | Epsilon: 0.740 | Goal Reached: YES
Episode:  40 | Steps: 726 | Total Reward: -2964.0 | Avg Reward: -3483.4 | Loss: 5.4935 | Epsilon: 0.669 | Goal Reached: YES
Episode:  50 | Steps: 169 | Total Reward: -605.0 | Avg Reward: -3753.8 | Loss: 3.0348 | Epsilon: 0.605 | Goal Reached: YES
Episode:  60 | Steps:  13 | Total Reward:   58.0 | Avg Reward: -2676.7 | Loss: 1.5874 | Epsilon: 0.547 | Goal Reached: YES
Episode:  70 | Steps: 299 | Total Reward: -1054.0 | Avg Rew